**Tech Stack Recommender**

In [189]:
import pandas as pd
import numpy as np
import math

data = pd.read_csv("/content/drive/MyDrive/raw_skills.csv")
df = pd.DataFrame(data)
df['skills'] = df['skills'].apply(lambda x: [skill.strip() for skill in x.split(',')])

unique_skills = list(df["skills"].explode().unique())
n_skills = len(unique_skills)
total_roles = len(df)
df.head()

,job_role,skills
0,Data Scientist,"[Python, SQL, Machine Learning, Data Analysis,..."
1,DevOps Engineer,"[AWS, Docker, Kubernetes, CI/CD, Linux, Git, T..."
2,Backend Developer,"[Java, Python, SQL, APIs, REST, Spring Boot, M..."
3,Cloud Architect,"[Cloud Computing, AWS, Azure, Automation, Terr..."
4,Frontend Developer,"[JavaScript, React, HTML, CSS, TypeScript, Git..."


**Calculating IDF**

In [190]:
idf = []
for i in unique_skills:
    count = 0
    for j in df['skills']:
        if i in j:
            count += 1
    idf.append(math.log(total_roles / count))

idf = np.array(idf)
print(idf)

[0.43078292 0.7985077  1.89711998 1.89711998 2.30258509 2.99573227
 2.99573227 2.99573227 1.60943791 1.38629436 1.60943791 1.89711998
 1.04982212 1.38629436 1.60943791 2.30258509 2.30258509 2.99573227
 2.99573227 2.99573227 2.99573227 1.89711998 2.30258509 1.2039728
 1.38629436 1.60943791 2.99573227 2.30258509 2.30258509 2.99573227
 2.99573227 2.99573227 1.89711998 2.99573227 2.30258509 2.30258509
 2.30258509 2.30258509 2.30258509 2.99573227 2.99573227 2.99573227
 2.99573227 2.99573227 2.99573227 2.99573227 2.99573227 2.30258509
 2.99573227 2.99573227 2.30258509 2.99573227 2.99573227 2.99573227
 2.30258509 2.99573227 2.99573227 2.99573227 2.99573227 2.99573227
 2.99573227 2.99573227 2.99573227 2.99573227 2.99573227 2.99573227
 2.99573227 2.99573227 2.99573227 2.99573227 2.99573227 2.99573227
 2.99573227 2.99573227 2.99573227 2.99573227 2.99573227 2.99573227
 2.99573227 2.99573227 2.99573227]


**Calculating tf_idf  & vectors**

In [191]:
def vector(skills):
  input_vector = [0] * n_skills
  for skill in skills:
      if skill in unique_skills:
        idx = unique_skills.index(skill)
        tf = 1 / len(skills)
        input_vector[idx] = tf * idf[idx]

  input_vector = np.array(input_vector)
  return input_vector

In [192]:
# input_vector
inputs = ['Docker', 'Kubernetes', 'Terraform']
unique_inputs = list(set(inputs))
input_vector = vector(unique_inputs)
print(input_vector)

[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.46209812 0.5364793  0.
 0.         0.         0.5364793  0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.        ]


In [193]:
db_vectors = []

for i in df['skills']:
  db_vectors.append(vector(i))

print(db_vectors[0])

[0.05384786 0.09981346 0.23714    0.23714    0.28782314 0.37446653
 0.37446653 0.37446653 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.        ]


**Cosine Similarity**

In [194]:
inp_vect_magnit = np.linalg.norm(input_vector)

db_vect_magnit = []
for i in db_vectors:
  db_vect_magnit.append(np.linalg.norm(i))

db_vect_magnit = np.array(db_vect_magnit)
ab = inp_vect_magnit * db_vect_magnit

print("Input vector magnitude: ", inp_vect_magnit)
print("Database vector magnitude: ", db_vect_magnit)
print("Product of both magnitudes: ", ab)

Input vector magnitude:  0.8883438300588894
Database vector magnitude:  [0.79300113 0.58118392 0.82863353 0.67334993 0.90158125 0.69520392
 0.64744162 0.99576053 0.6145574  0.77080454 0.65360008 0.80960334
 0.8027124  0.54113027 0.8552574  0.77630131 0.84429888 0.93873376
 0.89265357 0.51596046]
Product of both magnitudes:  [0.70445766 0.51629115 0.73611149 0.59816625 0.80091414 0.61758011
 0.57515077 0.88457772 0.54593828 0.68473945 0.58062159 0.71920613
 0.71308461 0.48070974 0.75976264 0.68962248 0.7500277  0.83391834
 0.7929833  0.45835029]


In [195]:
prodcut_of_vectors = []

for i in db_vectors:
    prodcut_of_vectors.append(np.dot(input_vector, i))

cosine_similarity = prodcut_of_vectors/ab
print(cosine_similarity)

[0.         0.57319021 0.         0.18043272 0.         0.
 0.         0.         0.14667501 0.         0.50968314 0.
 0.         0.39109728 0.         0.         0.         0.
 0.         0.64564819]


In [196]:
recomended = cosine_similarity.copy()
recomended = np.sort(recomended)[::-1]
recomended = recomended[:3]
print(recomended)

recom_index = []
cos_simi = list(cosine_similarity)
for i in recomended:
  if i in cos_simi:
    recom_index.append(cos_simi.index(i))

print(recom_index)

[0.64564819 0.57319021 0.50968314]
[19, 1, 10]


In [197]:
print("=" * 60)
print("Following are the recommended Roles based on the user input")
print("=" * 60)
print("\nUser Inputs: ", inputs)
print("Recommended Roles: ","\n")

for i in recom_index:
  print(df.iloc[i],'\n')

Following are the recommended Roles based on the user input

User Inputs:  ['Docker', 'Kubernetes', 'Terraform']
Recommended Roles:  

job_role                                    Platform Engineer
skills      [Kubernetes, Docker, CI/CD, Terraform, Cloud C...
Name: 19, dtype: object 

job_role                                      DevOps Engineer
skills      [AWS, Docker, Kubernetes, CI/CD, Linux, Git, T...
Name: 1, dtype: object 

job_role                                       Cloud Engineer
skills      [AWS, Azure, Google Cloud, Terraform, Automati...
Name: 10, dtype: object 

